This notebook is used to interpolate ERA5 to ERA5-land grid based on findings by Baas 2026 to create wind direction and wind speed histograms for 76 years if ERA5 reanalysis winds for the entire planet.

### Imports

In [ ]:
directory = '/g/data/k10/dn0361/create-era5-land-flux/' # path on NCI gadi

from glob import glob
import pickle
import itertools
import os
import time
from datetime import timedelta

import numpy as np
import shapely
import geopandas as gpd
from scipy.interpolate import CubicSpline

import cartopy.crs as ccrs
from matplotlib import pyplot as plt

import xarray as xr
from functools import partial
from xhistogram.xarray import histogram
import cf_xarray as cfxr

import warnings
warnings.filterwarnings('ignore')
import logging
logging.getLogger('flox').setLevel(logging.WARNING)

import dask
from dask.distributed import Client, wait
client = Client(threads_per_worker=1,memory_limit=0,silence_logs=logging.ERROR)
normalcores = client.ncores()
client

# session and environment on NCI
# conda env:analysis3-25.05
# Compute Size XXLarge

### Setup

In [ ]:
umchunk,uachunk,latchunk,lonchunk = 2**7,2**7,2**6,2**6

umbins = np.linspace(0,20,umchunk+1)
uabins = np.linspace(-np.pi,np.pi,uachunk+1)
years = np.arange(1950,2025+1)

### FOR ZARR SETUP ONLY ###
# ufiles,vfiles = list(itertools.chain.from_iterable([sorted(glob('/g/data/zz93/era5-land/reanalysis/u10/%d/*.nc'%year)) for year in years])),list(itertools.chain.from_iterable([sorted(glob('/g/data/zz93/era5-land/reanalysis/v10/%d/*.nc'%year)) for year in years]))
# files = np.stack((np.array(ufiles),np.array(vfiles))).T
# ds = xr.open_mfdataset(files[0])
# ds['um'] = (ds.u10**2+ds.v10**2)**0.5
# ds['ua'] = np.arctan2(ds.v10,ds.u10)
# dh = histogram(ds.um,ds.ua,bins=[umbins,uabins],dim=['time']).chunk({'latitude':latchunk,'longitude':lonchunk}).to_dataset()
# dh.to_zarr(directory+'era5-land_%d_%d_uam10_hist.zarr'%(years[0],years[-1]),compute=False,consolidated=False)

# get all files ERA5 files
ufiles,vfiles = list(itertools.chain.from_iterable([sorted(glob('/g/data/rt52/era5/single-levels/reanalysis/10u/%d/*.nc'%year)) for year in years])),list(itertools.chain.from_iterable([sorted(glob('/g/data/rt52/era5/single-levels/reanalysis/10v/%d/*.nc'%year)) for year in years]))
files = np.stack((np.array(ufiles),np.array(vfiles))).T

In [ ]:
dm = xr.open_dataset(directory+'mask.nc') # same grid as era5-land
latlonchunklandmask = np.ones((np.ceil(len(dm.latitude)/latchunk).astype(int),
                                np.ceil(len(dm.longitude)/lonchunk).astype(int)))
lonlatchunks = []
for i in range(latlonchunklandmask.shape[0]):
    for j in range(latlonchunklandmask.shape[1]):
        if dm.landmask.isel(latitude=slice(i*latchunk,(i+1)*latchunk),
                            longitude=slice(j*lonchunk,(j+1)*lonchunk)).isnull().all():
            latlonchunklandmask[i,j] = 0
        lonlatchunks.append([i,j])
lonlatchunks = np.array(lonlatchunks)
print(len(latlonchunklandmask[latlonchunklandmask==1]))
print(len(latlonchunklandmask[latlonchunklandmask==0]))
print(np.prod(latlonchunklandmask.shape))

lonlatchunksdone = np.array([np.array(file.split('/')[-1].split('.')[:2]).astype(int) for file in sorted(glob(directory+'era5-land_1950_2025_uam10_hist.zarr/histogram_um_ua/*.0.0'))])
lonlatchunkstodo = np.array([np.array(val) for val in sorted(list(set([tuple(val) for val in lonlatchunks])-set([tuple(val) for val in lonlatchunksdone])))])
print(int(latlonchunklandmask[*lonlatchunkstodo.T].sum()))
print(int(len(lonlatchunkstodo)-latlonchunklandmask[*lonlatchunkstodo.T].sum()))
print(len(lonlatchunkstodo))

In [ ]:
# function to handle era5 to era5-land interpolation
def make_preprocess(lats, lons, mask, edge):
    def preprocess(ds):
        if not edge:
            out = ds.sel(latitude=slice(lats[0]+buff,lats[-1]-buff),longitude=slice(lons[0]-buff,lons[-1]+buff)).chunk({"time": -1, "longitude": -1, "latitude": -1})
        else:
            out = xr.concat([ds.sel(latitude=slice(lats[0]+buff,lats[-1]-buff),
                                    longitude=slice(lons[0]-buff,lons[-1]+buff)),
                             ds.sel(latitude=slice(lats[0]+buff,lats[-1]-buff)).isel(longitude=0).assign_coords(longitude=180)],"longitude").chunk({"time": -1, "longitude": -1, "latitude": -1})
        out = out.interp(latitude=lats,longitude=lons)*mask
        return out
    return preprocess

### Calculate wind histograms

In [ ]:
%%time
countam = len(lonlatchunks)
counttm = int(latlonchunklandmask.sum())
countfm = countam-counttm
counta  = len(lonlatchunksdone)
countt  = int(latlonchunklandmask.sum()-latlonchunklandmask[*lonlatchunkstodo.T].sum())
countf  = counta-countt
counte  = 0
countr  = 0

# handle buffer and edge case
ridx = len(latlonchunklandmask.T)-1
buff=0.25+1e-6

errorlonlatchunks = []
time00 = time.time()
for lonlatchunk in lonlatchunkstodo:
    i,j = lonlatchunk
    counta += 1
    if latlonchunklandmask[i,j]:
        countt += 1
        t1=dm.isel(latitude=slice(i*latchunk,(i+1)*latchunk),longitude=slice(j*lonchunk,(j+1)*lonchunk)) # get era5-land extent
        ds=xr.open_mfdataset(files.flatten(), preprocess=make_preprocess(t1.latitude.values,t1.longitude.values,t1.landmask.values,j==ridx),parallel=True).chunk({'longitude':-1,'latitude':-1,'time':2**13})
        ds['um'] = (ds.u10**2+ds.v10**2)**0.5
        ds['ua'] = np.arctan2(ds.v10,ds.u10)
        dh = histogram(ds.um,ds.ua,bins=[umbins,uabins],dim=['time']).chunk({'latitude':latchunk,'longitude':lonchunk}).to_dataset()
    else:
        countf += 1
        dh = xr.open_mfdataset(directory+'era5-land_%d_%d_uam10_hist.zarr'%(years[0],years[-1]),
                               preprocess=lambda ds: ds.isel(latitude=slice(i*latchunk,(i+1)*latchunk),
                                                             longitude=slice(j*lonchunk,(j+1)*lonchunk)),
                               consolidated=False)            
        dh = dh.assign(histogram_um_ua=(dh.histogram_um_ua.dims,np.full(dh.histogram_um_ua.shape,0)))
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            dh.to_zarr(directory+'era5-land_%d_%d_uam10_hist.zarr'%(years[0],years[-1]),
                       region='auto',consolidated=False)
    except:
        counte += 1
        errorlonlatchunks.append(lonlatchunk)
    time1 = time.time()
    print('chunk %04d of %04d, %04d of %04d computed, %04d of %04d masked, %04d errors, %04d restarts %16s elapsed, estimated %16s remaining'%(counta,countam,countt,counttm,countf,countfm,counte,countr,
                                                                            str(timedelta(seconds=round(time1-time00))),
                                                                            str(timedelta(seconds=round((time1-time00)/countt*(counttm-countt))))),end='\r')
    if client.ncores()!=normalcores:
        countr += 1
        client.restart()
        normalcores = client.ncores()